# 89 — Index Geode metadata into the LBSSP shot catalog

This notebook imports the improved Jochen metadata workbook and the Geode file-time CSV into a SQLite-first catalog.

It is designed to be compatible with the nodal `90_*` notebook catalog schema so that later notebooks can query common shot positions across Geode, streamer, and nodal datasets.

The database is the source of truth. CSV exports are produced only as human-readable snapshots.

In [1]:
from pathlib import Path
import sqlite3
import json
import uuid
from datetime import datetime, timezone

import numpy as np
import pandas as pd

## 1. Configuration

In [2]:
PROJECT_ROOT = Path("/Volumes/tachyon/LBSSP_DATA")

METADATA_XLSX = PROJECT_ROOT / "metadata" / "jochen_field_notes_metadata_tables_with_geode_times.xlsx"
GEODE_FILE_TIMES_CSV = PROJECT_ROOT / "metadata" / "geode_file_times.csv"

# Fallback to uploaded/sandbox copies if running outside the project tree.
if not METADATA_XLSX.exists():
    METADATA_XLSX = Path("/mnt/data/jochen_field_notes_metadata_tables_with_geode_times.xlsx")
if not GEODE_FILE_TIMES_CSV.exists():
    GEODE_FILE_TIMES_CSV = Path("/mnt/data/geode_file_times.csv")

CATALOG_DIR = PROJECT_ROOT / "catalog"
CATALOG_DIR.mkdir(parents=True, exist_ok=True)

CATALOG_DB = CATALOG_DIR / "lbssp_shot_catalog.sqlite"
EXPORT_DIR = CATALOG_DIR / "catalog_exports"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

# Geode/Geometrics file start times are laptop-local times with no timezone offset.
LOCAL_TIMEZONE = "America/New_York"

NOTEBOOK_NAME = "89_index_geode_metadata_into_shot_catalog.ipynb"

print("METADATA_XLSX:", METADATA_XLSX)
print("GEODE_FILE_TIMES_CSV:", GEODE_FILE_TIMES_CSV)
print("CATALOG_DB:", CATALOG_DB)

# Delete prior Geode rows before importing, so reruns fix metadata parsing bugs cleanly.
REPLACE_EXISTING_GEODE_METADATA = True


METADATA_XLSX: /Volumes/tachyon/LBSSP_DATA/metadata/jochen_field_notes_metadata_tables_with_geode_times.xlsx
GEODE_FILE_TIMES_CSV: /Volumes/tachyon/LBSSP_DATA/metadata/geode_file_times.csv
CATALOG_DB: /Volumes/tachyon/LBSSP_DATA/catalog/lbssp_shot_catalog.sqlite


## 2. SQLite helpers and schema

In [3]:
def connect_catalog(db_path: Path) -> sqlite3.Connection:
    conn = sqlite3.connect(db_path)
    conn.execute("PRAGMA foreign_keys = ON;")
    conn.execute("PRAGMA journal_mode = WAL;")
    conn.execute("PRAGMA busy_timeout = 30000;")
    return conn


def table_exists(conn: sqlite3.Connection, table_name: str) -> bool:
    q = "SELECT name FROM sqlite_master WHERE type='table' AND name=?"
    return conn.execute(q, (table_name,)).fetchone() is not None


def write_df(conn: sqlite3.Connection, df: pd.DataFrame, table_name: str, if_exists: str = "append"):
    if df is None or len(df) == 0:
        print(f"{table_name}: no rows to write")
        return
    df.to_sql(table_name, conn, if_exists=if_exists, index=False)
    print(f"{table_name}: wrote {len(df)} rows")


def export_table(conn: sqlite3.Connection, table_name: str, export_dir: Path = EXPORT_DIR):
    if not table_exists(conn, table_name):
        return
    df = pd.read_sql(f"SELECT * FROM {table_name}", conn)
    out = export_dir / f"{table_name}.csv"
    df.to_csv(out, index=False)
    print(f"exported {table_name}: {out}")


def normalize_column_name(name) -> str:
    s = str(name).strip()
    s = s.replace(" ", "_").replace("-", "_").replace("/", "_")
    s = "".join(ch if ch.isalnum() or ch == "_" else "_" for ch in s)
    while "__" in s:
        s = s.replace("__", "_")
    return s.strip("_").lower()


def normalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out.columns = [normalize_column_name(c) for c in out.columns]
    return out


def df_extra_json(row: pd.Series, exclude: set) -> str:
    extra = {}
    for k, v in row.items():
        if k in exclude:
            continue
        if pd.isna(v):
            continue
        if isinstance(v, (pd.Timestamp, datetime)):
            extra[k] = v.isoformat()
        elif isinstance(v, (np.integer, np.floating)):
            extra[k] = float(v) if isinstance(v, np.floating) else int(v)
        else:
            extra[k] = str(v)
    return json.dumps(extra, sort_keys=True)


def local_naive_to_utc_iso(value, local_tz=LOCAL_TIMEZONE):
    if pd.isna(value) or value is None or str(value).strip() == "":
        return None
    ts = pd.to_datetime(value, errors="coerce")
    if pd.isna(ts):
        return None
    if ts.tzinfo is None:
        ts = ts.tz_localize(local_tz)
    return ts.tz_convert("UTC").isoformat()


def as_iso_or_none(value):
    if pd.isna(value) or value is None or str(value).strip() == "":
        return None
    ts = pd.to_datetime(value, errors="coerce")
    if pd.isna(ts):
        return str(value)
    return ts.isoformat()


def safe_int(value):
    if pd.isna(value):
        return None
    try:
        return int(value)
    except Exception:
        return None


def safe_float(value):
    if pd.isna(value):
        return None
    try:
        return float(value)
    except Exception:
        return None

In [4]:
SCHEMA_SQL = """
CREATE TABLE IF NOT EXISTS processing_runs (
    run_id TEXT PRIMARY KEY,
    notebook_name TEXT,
    run_time_utc TEXT,
    input_metadata_xlsx TEXT,
    input_geode_file_times_csv TEXT,
    catalog_db TEXT,
    parameters_json TEXT,
    notes TEXT
);

CREATE TABLE IF NOT EXISTS transect_crosswalk (
    final_transect TEXT,
    alias_or_field_note_name TEXT,
    description TEXT,
    status TEXT,
    run_id TEXT
);

CREATE TABLE IF NOT EXISTS acquisition_summary (
    acquisition_id TEXT PRIMARY KEY,
    date TEXT,
    day TEXT,
    transect TEXT,
    activity TEXT,
    geometry_or_key_points TEXT,
    source TEXT,
    source_page TEXT,
    confidence TEXT,
    review_status TEXT,
    start_time_local TEXT,
    end_time_local TEXT,
    start_time_utc TEXT,
    end_time_utc TEXT,
    extra_json TEXT,
    run_id TEXT
);

CREATE TABLE IF NOT EXISTS survey_issues (
    issue_id TEXT PRIMARY KEY,
    sheet TEXT,
    row_or_scope TEXT,
    description TEXT,
    recommended_action TEXT,
    priority TEXT,
    run_id TEXT
);

CREATE TABLE IF NOT EXISTS geode_file_times (
    file_no INTEGER,
    datfile TEXT,
    datfile_stem TEXT,
    geode_folder TEXT,
    geode_folder_date TEXT,
    geode_file_path TEXT,
    read_ok INTEGER,
    read_format TEXT,
    error TEXT,
    n_traces INTEGER,
    sampling_rate_hz REAL,
    npts_first_trace INTEGER,
    duration_s_first_trace REAL,
    geode_laptop_starttime_local TEXT,
    geode_laptop_starttime_utc TEXT,
    geode_laptop_starttime_epoch REAL,
    geode_time_source TEXT,
    run_id TEXT
);

CREATE TABLE IF NOT EXISTS shot_events (
    event_id TEXT PRIMARY KEY,
    instrument_system TEXT,
    line TEXT,
    transect TEXT,
    survey TEXT,
    survey_type TEXT,
    shot_no INTEGER,
    file_no INTEGER,
    source_x_m REAL,
    source_type TEXT,
    n_blows INTEGER,
    n_shots INTEGER,
    operator TEXT,
    plate_type TEXT,
    shot_time_local TEXT,
    shot_time_utc TEXT,
    receiver_first_m REAL,
    receiver_last_m REAL,
    receiver_spacing_m REAL,
    nominal_shot_spacing_m REAL,
    geode_read_ok INTEGER,
    geode_read_format TEXT,
    geode_n_traces INTEGER,
    geode_sampling_rate_hz REAL,
    geode_duration_s_first_trace REAL,
    geode_file_path TEXT,
    geode_folder TEXT,
    geode_match_status TEXT,
    geode_match_note TEXT,
    source_page TEXT,
    confidence TEXT,
    review_status TEXT,
    comment TEXT,
    metadata_source_sheet TEXT,
    extra_json TEXT,
    run_id TEXT
);

CREATE TABLE IF NOT EXISTS receiver_geometry (
    geometry_id TEXT,
    event_id TEXT,
    instrument_system TEXT,
    line TEXT,
    transect TEXT,
    survey TEXT,
    receiver_index INTEGER,
    receiver_x_m REAL,
    receiver_y_m REAL,
    receiver_spacing_m REAL,
    component TEXT,
    sample_rate_hz REAL,
    geometry_status TEXT,
    geometry_note TEXT,
    run_id TEXT
);

CREATE TABLE IF NOT EXISTS shot_gather_files (
    gather_file_id TEXT PRIMARY KEY,
    event_id TEXT,
    gather_id TEXT,
    instrument_system TEXT,
    line TEXT,
    transect TEXT,
    survey TEXT,
    component TEXT,
    file_type TEXT,
    file_path TEXT,
    source_file_no INTEGER,
    n_traces INTEGER,
    n_receivers INTEGER,
    sample_rate_hz REAL,
    duration_s REAL,
    processing_level TEXT,
    geometry_id TEXT,
    run_id TEXT
);

CREATE TABLE IF NOT EXISTS trace_index (
    trace_index_id TEXT PRIMARY KEY,
    event_id TEXT,
    gather_id TEXT,
    instrument_system TEXT,
    line TEXT,
    transect TEXT,
    survey TEXT,
    source_x_m REAL,
    receiver_index INTEGER,
    receiver_x_m REAL,
    offset_m REAL,
    component TEXT,
    file_path TEXT,
    trace_number_in_file INTEGER,
    sample_rate_hz REAL,
    npts INTEGER,
    amplitude_scale REAL,
    run_id TEXT
);

CREATE TABLE IF NOT EXISTS metadata_raw_sheets (
    raw_row_id TEXT PRIMARY KEY,
    source_sheet TEXT,
    row_number INTEGER,
    row_json TEXT,
    run_id TEXT
);

CREATE INDEX IF NOT EXISTS idx_shot_events_line_x ON shot_events(line, source_x_m);
CREATE INDEX IF NOT EXISTS idx_shot_events_file_no ON shot_events(file_no);
CREATE INDEX IF NOT EXISTS idx_shot_events_time ON shot_events(shot_time_utc);
CREATE INDEX IF NOT EXISTS idx_gather_files_event ON shot_gather_files(event_id);
CREATE INDEX IF NOT EXISTS idx_trace_index_event ON trace_index(event_id);
"""
conn = connect_catalog(CATALOG_DB)
conn.executescript(SCHEMA_SQL)
conn.commit()
print("Schema ready")

Schema ready


## 3. Register this processing run

In [5]:
RUN_ID = f"run_{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}_{uuid.uuid4().hex[:8]}"

params = {
    "local_timezone": LOCAL_TIMEZONE,
    "metadata_xlsx": str(METADATA_XLSX),
    "geode_file_times_csv": str(GEODE_FILE_TIMES_CSV),
}

run_df = pd.DataFrame([{
    "run_id": RUN_ID,
    "notebook_name": NOTEBOOK_NAME,
    "run_time_utc": datetime.now(timezone.utc).isoformat(),
    "input_metadata_xlsx": str(METADATA_XLSX),
    "input_geode_file_times_csv": str(GEODE_FILE_TIMES_CSV),
    "catalog_db": str(CATALOG_DB),
    "parameters_json": json.dumps(params, sort_keys=True),
    "notes": "Index improved Geode field metadata and Geode file timing information."
}])

write_df(conn, run_df, "processing_runs", if_exists="append")
RUN_ID

# Optional cleanup for reruns. This prevents duplicate primary-key errors and
# lets this notebook fix earlier metadata parsing mistakes, such as incorrect
# T1 streamer line labels like T10/T11.
if REPLACE_EXISTING_GEODE_METADATA:
    cleanup_sql = [
        "DELETE FROM trace_index WHERE instrument_system = 'geode'",
        "DELETE FROM shot_gather_files WHERE instrument_system = 'geode'",
        "DELETE FROM receiver_geometry WHERE instrument_system = 'geode'",
        "DELETE FROM shot_events WHERE instrument_system = 'geode'",
        "DELETE FROM geode_file_times",
        "DELETE FROM transect_crosswalk",
        "DELETE FROM acquisition_summary",
        "DELETE FROM survey_issues",
        "DELETE FROM metadata_raw_sheets",
    ]
    for sql in cleanup_sql:
        try:
            conn.execute(sql)
        except Exception as e:
            print(f"Cleanup skipped/failed for: {sql} -> {e}")
    conn.commit()
    print("Removed previous Geode metadata/catalog rows before import.")

processing_runs: wrote 1 rows
Removed previous Geode metadata/catalog rows before import.


## 4. Read workbook and Geode time CSV

In [6]:
xl = pd.ExcelFile(METADATA_XLSX)
sheet_names = xl.sheet_names
print(sheet_names)

sheets = {name: normalize_columns(pd.read_excel(METADATA_XLSX, sheet_name=name)) for name in sheet_names}
geode_times = normalize_columns(pd.read_csv(GEODE_FILE_TIMES_CSV))

for name, df in sheets.items():
    print(f"{name:35s} {df.shape}")
print("geode_file_times_csv", geode_times.shape)

['README', 'Transect_Crosswalk', 'Acquisition_Summary', 'T1_1m_Refraction', 'T1_2m_Refraction', 'T1_Streamer_MASW', 'T1A_Streamer_MASW', 'T3_1m_Refraction', 'T4_1m_Refraction', 'Review_Issues', 'Geode_Time_Extraction_Summary']
README                              (6, 2)
Transect_Crosswalk                  (4, 4)
Acquisition_Summary                 (14, 12)
T1_1m_Refraction                    (46, 29)
T1_2m_Refraction                    (42, 29)
T1_Streamer_MASW                    (83, 25)
T1A_Streamer_MASW                   (87, 25)
T3_1m_Refraction                    (39, 29)
T4_1m_Refraction                    (30, 29)
Review_Issues                       (6, 6)
Geode_Time_Extraction_Summary       (13, 5)
geode_file_times_csv (326, 17)


## 5. Store raw workbook rows for provenance

In [7]:
raw_rows = []
for sheet_name, df in sheets.items():
    for i, row in df.iterrows():
        clean = {}
        for k, v in row.items():
            if pd.isna(v):
                continue
            if isinstance(v, (pd.Timestamp, datetime)):
                clean[k] = v.isoformat()
            else:
                clean[k] = str(v)
        raw_rows.append({
            "raw_row_id": f"{RUN_ID}_{normalize_column_name(sheet_name)}_{i:06d}",
            "source_sheet": sheet_name,
            "row_number": int(i),
            "row_json": json.dumps(clean, sort_keys=True),
            "run_id": RUN_ID,
        })

raw_df = pd.DataFrame(raw_rows)
write_df(conn, raw_df, "metadata_raw_sheets", if_exists="append")
raw_df.head()

metadata_raw_sheets: wrote 370 rows


,raw_row_id,source_sheet,row_number,row_json,run_id
0,run_20260612T200044Z_bdd2ebb7_readme_000000,README,0,"{""jochen_field_notes_metadata_extraction"": ""LB...",run_20260612T200044Z_bdd2ebb7
1,run_20260612T200044Z_bdd2ebb7_readme_000001,README,1,"{""jochen_field_notes_metadata_extraction"": ""20...",run_20260612T200044Z_bdd2ebb7
2,run_20260612T200044Z_bdd2ebb7_readme_000002,README,2,"{""jochen_field_notes_metadata_extraction"": ""Ma...",run_20260612T200044Z_bdd2ebb7
3,run_20260612T200044Z_bdd2ebb7_readme_000003,README,3,"{""jochen_field_notes_metadata_extraction"": ""Ha...",run_20260612T200044Z_bdd2ebb7
4,run_20260612T200044Z_bdd2ebb7_readme_000004,README,4,"{""jochen_field_notes_metadata_extraction"": ""T1...",run_20260612T200044Z_bdd2ebb7


## 6. Import crosswalk, acquisition summary, review issues, and Geode file times

In [8]:
if "Transect_Crosswalk" in sheets:
    df = sheets["Transect_Crosswalk"].copy()
    df["run_id"] = RUN_ID
    write_df(conn, df, "transect_crosswalk", if_exists="append")
    display(df)

if "Acquisition_Summary" in sheets:
    src = sheets["Acquisition_Summary"].copy()
    rows = []
    for i, r in src.iterrows():
        rows.append({
            "acquisition_id": f"ACQ_{RUN_ID}_{i+1:04d}",
            "date": as_iso_or_none(r.get("date")),
            "day": r.get("day"),
            "transect": r.get("transect"),
            "activity": r.get("activity"),
            "geometry_or_key_points": r.get("geometry_or_key_points"),
            "source": r.get("source"),
            "source_page": r.get("source_page"),
            "confidence": r.get("confidence"),
            "review_status": r.get("review_status"),
            "start_time_local": as_iso_or_none(r.get("start_time")),
            "end_time_local": as_iso_or_none(r.get("end_time")),
            "start_time_utc": local_naive_to_utc_iso(r.get("start_time")),
            "end_time_utc": local_naive_to_utc_iso(r.get("end_time")),
            "extra_json": df_extra_json(r, {
                "date","day","transect","activity","geometry_or_key_points","source","source_page",
                "confidence","review_status","start_time","end_time"
            }),
            "run_id": RUN_ID,
        })
    acquisition_df = pd.DataFrame(rows)
    write_df(conn, acquisition_df, "acquisition_summary", if_exists="append")
    display(acquisition_df.head())

if "Review_Issues" in sheets:
    df = sheets["Review_Issues"].copy()
    # Issue IDs are reused across reruns, so make stored IDs unique to this run.
    if "issue_id" in df.columns:
        df["issue_id"] = df["issue_id"].apply(lambda x: f"{RUN_ID}_{x}")
    df["run_id"] = RUN_ID
    write_df(conn, df, "survey_issues", if_exists="append")
    display(df.head())

gt = geode_times.copy()
gt["file_no"] = gt["file_no"].apply(safe_int)
gt["read_ok"] = gt["read_ok"].fillna(False).astype(bool).astype(int)
gt["n_traces"] = gt["n_traces"].apply(safe_int)
gt["npts_first_trace"] = gt["npts_first_trace"].apply(safe_int)
local_source = gt["geode_laptop_starttime_iso"] if "geode_laptop_starttime_iso" in gt else gt.get("geode_laptop_starttime")
gt["geode_laptop_starttime_local"] = local_source.apply(as_iso_or_none)
gt["geode_laptop_starttime_utc"] = local_source.apply(local_naive_to_utc_iso)

gt_db = pd.DataFrame({
    "file_no": gt.get("file_no"),
    "datfile": gt.get("datfile"),
    "datfile_stem": gt.get("datfile_stem"),
    "geode_folder": gt.get("geode_folder"),
    "geode_folder_date": gt.get("geode_folder_date"),
    "geode_file_path": gt.get("geode_file_path"),
    "read_ok": gt.get("read_ok"),
    "read_format": gt.get("read_format"),
    "error": gt.get("error"),
    "n_traces": gt.get("n_traces"),
    "sampling_rate_hz": gt.get("sampling_rate_hz"),
    "npts_first_trace": gt.get("npts_first_trace"),
    "duration_s_first_trace": gt.get("duration_s_first_trace"),
    "geode_laptop_starttime_local": gt.get("geode_laptop_starttime_local"),
    "geode_laptop_starttime_utc": gt.get("geode_laptop_starttime_utc"),
    "geode_laptop_starttime_epoch": gt.get("geode_laptop_starttime_epoch"),
    "geode_time_source": gt.get("geode_time_source"),
    "run_id": RUN_ID,
})
write_df(conn, gt_db, "geode_file_times", if_exists="append")
display(gt_db.head())

transect_crosswalk: wrote 4 rows


,final_transect,alias_or_field_note_name,description,status,run_id
0,T1,central main transect,Main cave-target transect,confirmed by Glenn,run_20260612T200044Z_bdd2ebb7
1,T1A,T2 in Glenn notes,Western transect named T1A by Rocco,use both names in metadata until filenames rec...,run_20260612T200044Z_bdd2ebb7
2,T3,eastern transect,Eastern transect with refraction and nodal dep...,confirmed by Glenn,run_20260612T200044Z_bdd2ebb7
3,T4,control line,Control refraction line; no nodal deployment,confirmed by Glenn,run_20260612T200044Z_bdd2ebb7


acquisition_summary: wrote 14 rows


,acquisition_id,date,day,transect,activity,geometry_or_key_points,source,source_page,confidence,review_status,start_time_local,end_time_local,start_time_utc,end_time_utc,extra_json,run_id
0,ACQ_run_20260612T200044Z_bdd2ebb7_0001,2026-05-16T00:00:00,Day 1,T1,Line setup,Alternating yellow/green flags every 4 m along...,Glenn notes + Jochen sketch,LBSS3-LBSS5,Medium,CHECK,None,None,None,None,{},run_20260612T200044Z_bdd2ebb7
1,ACQ_run_20260612T200044Z_bdd2ebb7_0002,2026-05-16T00:00:00,Day 1,T1,MASW/streamer,"24-channel streamer, nominal 5 ft spacing; PEG...",Jochen notes,LBSS5-LBSS9,Medium,PARTIAL -- updated jb,2026-05-16T17:13:00,2026-05-16T21:48:00,2026-05-16T21:13:00+00:00,2026-05-17T01:48:00+00:00,{},run_20260612T200044Z_bdd2ebb7
2,ACQ_run_20260612T200044Z_bdd2ebb7_0003,2026-05-16T00:00:00,Day 1,T1,SmartSolo Survey1,"36 nodes; central 4 m spacing, outer 8 m spaci...",Glenn notes + spreadsheet,spreadsheet survey1,High,OK,None,None,None,None,{},run_20260612T200044Z_bdd2ebb7
3,ACQ_run_20260612T200044Z_bdd2ebb7_0004,2026-05-17T00:00:00,Day 2,T1A,MASW/streamer,"Western transect, referred to as T2 in Glenn n...",Glenn + Jochen notes,LBSS10-LBSS14,Medium,PARTIAL - updated jb,2026-05-17T14:00:04,2026-05-17T19:05:47,2026-05-17T18:00:04+00:00,2026-05-17T23:05:47+00:00,{},run_20260612T200044Z_bdd2ebb7
4,ACQ_run_20260612T200044Z_bdd2ebb7_0005,2026-05-17T00:00:00,Day 2,T1,SmartSolo Survey2,Moved 5 north-end and 4 south-end nodes to den...,Glenn notes + spreadsheet,spreadsheet survey2,High,OK,None,None,None,None,{},run_20260612T200044Z_bdd2ebb7


survey_issues: wrote 6 rows


,issue_id,sheet,row_or_scope,description,recommended_action,priority,run_id
0,run_20260612T200044Z_bdd2ebb7_ISSUE_001,T1_1m_Refraction,shots 1-5,Early repeated shots at 82.5 m include plastic...,Check LBSS16 image before using for quantitati...,High,run_20260612T200044Z_bdd2ebb7
1,run_20260612T200044Z_bdd2ebb7_ISSUE_002,T1_1m_Refraction,shots 11-12,Blow count and operator uncertain.,Verify against original LBSS16 page; JB update...,Medium,run_20260612T200044Z_bdd2ebb7
2,run_20260612T200044Z_bdd2ebb7_ISSUE_003,T1_2m_Refraction,shot 1,Operator field hard to decipher.,Verify against LBSS18; JB update: verified,Medium,run_20260612T200044Z_bdd2ebb7
3,run_20260612T200044Z_bdd2ebb7_ISSUE_004,T1A_Streamer_MASW,all,Streamer source type and some line-end positio...,Review LBSS11-LBSS14 before generating final S...,High,run_20260612T200044Z_bdd2ebb7
4,run_20260612T200044Z_bdd2ebb7_ISSUE_005,Acquisition_Summary,T4,T4 control-line geometry still based on Glenn ...,Add T4 shot table when notes are available/leg...,High,run_20260612T200044Z_bdd2ebb7


geode_file_times: wrote 326 rows


,file_no,datfile,datfile_stem,geode_folder,geode_folder_date,geode_file_path,read_ok,read_format,error,n_traces,sampling_rate_hz,npts_first_trace,duration_s_first_trace,geode_laptop_starttime_local,geode_laptop_starttime_utc,geode_laptop_starttime_epoch,geode_time_source,run_id
0,1001.0,1001.dat,1001,LBS_051626,2026-05-16,/Volumes/tachyon/LBSSP_DATA/GEODE_DATA/LBS_051...,1,SEG2,NaN,24.0,2000.0,4000.0,1.9995,2026-05-16T13:12:26,2026-05-16T17:12:26+00:00,1.778937e+09,obspy_trace_stats_starttime,run_20260612T200044Z_bdd2ebb7
1,1002.0,1002.dat,1002,LBS_051626,2026-05-16,/Volumes/tachyon/LBSSP_DATA/GEODE_DATA/LBS_051...,1,SEG2,NaN,24.0,2000.0,4000.0,1.9995,2026-05-16T14:20:53,2026-05-16T18:20:53+00:00,1.778941e+09,obspy_trace_stats_starttime,run_20260612T200044Z_bdd2ebb7
2,1003.0,1003.dat,1003,LBS_051626,2026-05-16,/Volumes/tachyon/LBSSP_DATA/GEODE_DATA/LBS_051...,1,SEG2,NaN,24.0,2000.0,4000.0,1.9995,2026-05-16T14:24:44,2026-05-16T18:24:44+00:00,1.778941e+09,obspy_trace_stats_starttime,run_20260612T200044Z_bdd2ebb7
3,1004.0,1004.dat,1004,LBS_051626,2026-05-16,/Volumes/tachyon/LBSSP_DATA/GEODE_DATA/LBS_051...,1,SEG2,NaN,24.0,2000.0,4000.0,1.9995,2026-05-16T14:28:19,2026-05-16T18:28:19+00:00,1.778942e+09,obspy_trace_stats_starttime,run_20260612T200044Z_bdd2ebb7
4,1005.0,1005.dat,1005,LBS_051626,2026-05-16,/Volumes/tachyon/LBSSP_DATA/GEODE_DATA/LBS_051...,1,SEG2,NaN,24.0,2000.0,4000.0,1.9995,2026-05-16T14:32:02,2026-05-16T18:32:02+00:00,1.778942e+09,obspy_trace_stats_starttime,run_20260612T200044Z_bdd2ebb7


## 7. Build Geode shot event rows from survey sheets

In [9]:
SURVEY_SHEETS = [
    "T1_1m_Refraction",
    "T1_2m_Refraction",
    "T1_Streamer_MASW",
    "T1A_Streamer_MASW",
    "T3_1m_Refraction",
    "T4_1m_Refraction",
]

def infer_survey_type(sheet_name, survey_value=None):
    s = f"{sheet_name} {survey_value or ''}".lower()
    if "streamer" in s or "masw" in s:
        return "streamer_masw"
    if "refraction" in s:
        return "refraction"
    return "unknown"


def infer_source_type(sheet_name, row):
    if pd.notna(row.get("source_type")):
        return str(row.get("source_type"))
    s = f"{sheet_name} {row.get('survey','')}".lower()
    if "streamer" in s or "masw" in s:
        return "PEG"
    return "hammer"


def get_source_x(row):
    if pd.notna(row.get("source_position_m")):
        return safe_float(row.get("source_position_m"))
    if pd.notna(row.get("shot_location_m")):
        return safe_float(row.get("shot_location_m"))
    return None


def get_comment(row):
    parts = []
    for k in ["comment", "comments", "geode_match_note"]:
        v = row.get(k)
        if pd.notna(v):
            parts.append(str(v))
    return " | ".join(parts) if parts else None


def infer_line_from_sheet(sheet_name, transect=None):
    """Return canonical line from sheet name first, then transect.

    This deliberately avoids interpreting numeric row/transect values as T10, T11, etc.
    For example, all rows from T1_Streamer_MASW are line T1, even when the
    event/file number is F1010 or a row field contains 10/11/12.
    """
    sheet = str(sheet_name)
    tr = None if transect is None or pd.isna(transect) else str(transect).strip()
    if sheet.startswith("T1A_"):
        return "T1A"
    if sheet.startswith("T1_"):
        return "T1"
    if sheet.startswith("T3_"):
        return "T3"
    if sheet.startswith("T4_"):
        return "T4"
    if tr in {"T1A", "T2"}:
        return "T1A"
    if tr in {"T1", "T3", "T4"}:
        return tr
    return tr


shot_rows = []

for sheet in SURVEY_SHEETS:
    if sheet not in sheets:
        print(f"Missing sheet: {sheet}")
        continue

    df = sheets[sheet].copy()

    for _, r in df.iterrows():
        file_no = safe_int(r.get("file_no"))
        shot_no = safe_int(r.get("shot_no"))
        transect = str(r.get("transect")) if pd.notna(r.get("transect")) else None
        line = infer_line_from_sheet(sheet, transect)

        survey = r.get("survey") if pd.notna(r.get("survey")) else sheet
        survey_type = infer_survey_type(sheet, survey)

        if file_no is not None:
            event_id = f"GEODE_{normalize_column_name(sheet).upper()}_F{file_no:04d}"
        else:
            event_id = f"GEODE_{normalize_column_name(sheet).upper()}_S{shot_no or 0:04d}_{uuid.uuid4().hex[:6]}"

        local_time = r.get("geode_laptop_starttime_iso")
        if pd.isna(local_time):
            local_time = r.get("geode_laptop_starttime")

        exclude = {
            "shot_no","file_no","transect","survey","source_position_m","shot_location_m",
            "source_type","n_blows","n_shots","operator","plate_type","comment","comments",
            "geode_laptop_starttime","geode_laptop_starttime_iso","receiver_first_m","receiver_last_m",
            "receiver_spacing_m","nominal_shot_spacing_m","geode_read_ok","geode_read_format",
            "geode_n_traces","geode_sampling_rate_hz","geode_duration_s_first_trace","geode_file_path",
            "geode_folder","geode_match_status","geode_match_note","source_page","confidence","review_status"
        }

        shot_rows.append({
            "event_id": event_id,
            "instrument_system": "geode",
            "line": line,
            "transect": transect,
            "survey": survey,
            "survey_type": survey_type,
            "shot_no": shot_no,
            "file_no": file_no,
            "source_x_m": get_source_x(r),
            "source_type": infer_source_type(sheet, r),
            "n_blows": safe_int(r.get("n_blows")),
            "n_shots": safe_int(r.get("n_shots")),
            "operator": r.get("operator") if pd.notna(r.get("operator")) else None,
            "plate_type": r.get("plate_type") if pd.notna(r.get("plate_type")) else None,
            "shot_time_local": as_iso_or_none(local_time),
            "shot_time_utc": local_naive_to_utc_iso(local_time),
            "receiver_first_m": safe_float(r.get("receiver_first_m")),
            "receiver_last_m": safe_float(r.get("receiver_last_m")),
            "receiver_spacing_m": safe_float(r.get("receiver_spacing_m")),
            "nominal_shot_spacing_m": safe_float(r.get("nominal_shot_spacing_m")),
            "geode_read_ok": int(bool(r.get("geode_read_ok"))) if pd.notna(r.get("geode_read_ok")) else None,
            "geode_read_format": r.get("geode_read_format") if pd.notna(r.get("geode_read_format")) else None,
            "geode_n_traces": safe_int(r.get("geode_n_traces")),
            "geode_sampling_rate_hz": safe_float(r.get("geode_sampling_rate_hz")),
            "geode_duration_s_first_trace": safe_float(r.get("geode_duration_s_first_trace")),
            "geode_file_path": r.get("geode_file_path") if pd.notna(r.get("geode_file_path")) else None,
            "geode_folder": r.get("geode_folder") if pd.notna(r.get("geode_folder")) else None,
            "geode_match_status": r.get("geode_match_status") if pd.notna(r.get("geode_match_status")) else None,
            "geode_match_note": r.get("geode_match_note") if pd.notna(r.get("geode_match_note")) else None,
            "source_page": r.get("source_page") if pd.notna(r.get("source_page")) else None,
            "confidence": r.get("confidence") if pd.notna(r.get("confidence")) else None,
            "review_status": r.get("review_status") if pd.notna(r.get("review_status")) else None,
            "comment": get_comment(r),
            "metadata_source_sheet": sheet,
            "extra_json": df_extra_json(r, exclude),
            "run_id": RUN_ID,
        })

shot_events_df = pd.DataFrame(shot_rows)

display(shot_events_df.head())
print(shot_events_df.shape)
display(shot_events_df.groupby(["metadata_source_sheet", "survey_type"]).size().reset_index(name="n"))

# Sanity check: line values should be canonical, not T10/T11/etc.
bad_line_events = shot_events_df[~shot_events_df["line"].isin(["T1", "T1A", "T3", "T4"])]
if len(bad_line_events):
    print("WARNING: unexpected line labels detected")
    display(bad_line_events[["event_id", "metadata_source_sheet", "line", "transect", "file_no", "shot_no"]].head(20))


,event_id,instrument_system,line,transect,survey,survey_type,shot_no,file_no,source_x_m,source_type,...,geode_folder,geode_match_status,geode_match_note,source_page,confidence,review_status,comment,metadata_source_sheet,extra_json,run_id
0,GEODE_T1_1M_REFRACTION_F3001,geode,T1,T1,T1_1m_refraction,refraction,1,3001,82.5,hammer,...,None,no_matching_dat_file,No .dat file with this file_no was found under...,LBSS16,Medium,CHECK,plastic plate broke? early test | No .dat file...,T1_1m_Refraction,{},run_20260612T200044Z_bdd2ebb7
1,GEODE_T1_1M_REFRACTION_F3002,geode,T1,T1,T1_1m_refraction,refraction,2,3002,82.5,hammer,...,None,no_matching_dat_file,No .dat file with this file_no was found under...,LBSS16,Medium,CHECK,plastic plate broke? early test | No .dat file...,T1_1m_Refraction,{},run_20260612T200044Z_bdd2ebb7
2,GEODE_T1_1M_REFRACTION_F3003,geode,T1,T1,T1_1m_refraction,refraction,3,3003,82.5,hammer,...,None,no_matching_dat_file,No .dat file with this file_no was found under...,LBSS16,Medium,CHECK,test / decided to use metal plate | No .dat fi...,T1_1m_Refraction,{},run_20260612T200044Z_bdd2ebb7
3,GEODE_T1_1M_REFRACTION_F3004,geode,T1,T1,T1_1m_refraction,refraction,4,3004,82.5,hammer,...,None,no_matching_dat_file,No .dat file with this file_no was found under...,LBSS16,Low,CHECK,ambiguous early plate-test row | No .dat file ...,T1_1m_Refraction,{},run_20260612T200044Z_bdd2ebb7
4,GEODE_T1_1M_REFRACTION_F3005,geode,T1,T1,T1_1m_refraction,refraction,5,3005,82.5,hammer,...,LBSSP_051826,matched,single file_no match,LBSS16,Medium,CHECK,delayed; first standard-looking shot | single ...,T1_1m_Refraction,"{""geode_folder_date"": ""2026-05-18T00:00:00"", ""...",run_20260612T200044Z_bdd2ebb7


(327, 36)


,metadata_source_sheet,survey_type,n
0,T1A_Streamer_MASW,streamer_masw,87
1,T1_1m_Refraction,refraction,46
2,T1_2m_Refraction,refraction,42
3,T1_Streamer_MASW,streamer_masw,83
4,T3_1m_Refraction,refraction,39
5,T4_1m_Refraction,refraction,30


## 8. Build receiver geometry, gather file, and trace-index rows

In [10]:
def build_geode_receiver_geometry_for_event(r):
    rows = []

    event_id = r["event_id"]
    n_traces = safe_int(r.get("geode_n_traces"))
    sr = safe_float(r.get("geode_sampling_rate_hz"))
    geometry_id = f"GEOM_{event_id}"

    first = safe_float(r.get("receiver_first_m"))
    spacing = safe_float(r.get("receiver_spacing_m"))

    if first is not None and spacing is not None and n_traces is not None:
        for i in range(n_traces):
            x = first + i * spacing
            rows.append({
                "geometry_id": geometry_id,
                "event_id": event_id,
                "instrument_system": "geode",
                "line": r.get("line"),
                "transect": r.get("transect"),
                "survey": r.get("survey"),
                "receiver_index": i + 1,
                "receiver_x_m": x,
                "receiver_y_m": None,
                "receiver_spacing_m": spacing,
                "component": "Z",
                "sample_rate_hz": sr,
                "geometry_status": "expanded_from_refraction_metadata",
                "geometry_note": f"receiver_first_m={first}, receiver_spacing_m={spacing}, n_traces={n_traces}",
                "run_id": RUN_ID,
            })
    else:
        rows.append({
            "geometry_id": geometry_id,
            "event_id": event_id,
            "instrument_system": "geode",
            "line": r.get("line"),
            "transect": r.get("transect"),
            "survey": r.get("survey"),
            "receiver_index": None,
            "receiver_x_m": None,
            "receiver_y_m": None,
            "receiver_spacing_m": None,
            "component": "Z",
            "sample_rate_hz": sr,
            "geometry_status": "streamer_geometry_pending",
            "geometry_note": "Streamer receiver positions require roll/spread geometry; not inferred here.",
            "run_id": RUN_ID,
        })

    return rows


geom_rows = []
gather_file_rows = []
trace_rows = []

for _, r in shot_events_df.iterrows():
    event_id = r["event_id"]
    file_no = safe_int(r.get("file_no"))
    file_path = r.get("geode_file_path")
    geometry_id = f"GEOM_{event_id}"
    gather_id = f"GATHER_{event_id}_RAW"

    geom = build_geode_receiver_geometry_for_event(r)
    geom_rows.extend(geom)

    n_traces = safe_int(r.get("geode_n_traces"))
    sr = safe_float(r.get("geode_sampling_rate_hz"))
    duration = safe_float(r.get("geode_duration_s_first_trace"))

    if file_path is not None:
        gather_file_rows.append({
            "gather_file_id": f"GF_{event_id}_RAW_DAT",
            "event_id": event_id,
            "gather_id": gather_id,
            "instrument_system": "geode",
            "line": r.get("line"),
            "transect": r.get("transect"),
            "survey": r.get("survey"),
            "component": "Z",
            "file_type": r.get("geode_read_format") or "SEG2_DAT",
            "file_path": file_path,
            "source_file_no": file_no,
            "n_traces": n_traces,
            "n_receivers": n_traces,
            "sample_rate_hz": sr,
            "duration_s": duration,
            "processing_level": "RAW",
            "geometry_id": geometry_id,
            "run_id": RUN_ID,
        })

    safe_geom = [g for g in geom if g.get("receiver_x_m") is not None]
    for g in safe_geom:
        rx = safe_float(g["receiver_x_m"])
        sx = safe_float(r.get("source_x_m"))
        npts = None if duration is None or sr is None else int(round(duration * sr))
        trace_rows.append({
            "trace_index_id": f"TR_{event_id}_{int(g['receiver_index']):03d}",
            "event_id": event_id,
            "gather_id": gather_id,
            "instrument_system": "geode",
            "line": r.get("line"),
            "transect": r.get("transect"),
            "survey": r.get("survey"),
            "source_x_m": sx,
            "receiver_index": g["receiver_index"],
            "receiver_x_m": rx,
            "offset_m": None if sx is None or rx is None else rx - sx,
            "component": "Z",
            "file_path": file_path,
            "trace_number_in_file": g["receiver_index"],
            "sample_rate_hz": sr,
            "npts": npts,
            "amplitude_scale": None,
            "run_id": RUN_ID,
        })

receiver_geometry_df = pd.DataFrame(geom_rows)
shot_gather_files_df = pd.DataFrame(gather_file_rows)
trace_index_df = pd.DataFrame(trace_rows)

print("receiver_geometry", receiver_geometry_df.shape)
print("shot_gather_files", shot_gather_files_df.shape)
print("trace_index", trace_index_df.shape)

display(receiver_geometry_df.head())
display(shot_gather_files_df.head())
display(trace_index_df.head())

receiver_geometry (7285, 15)
shot_gather_files (185, 18)
trace_index (7056, 18)


,geometry_id,event_id,instrument_system,line,transect,survey,receiver_index,receiver_x_m,receiver_y_m,receiver_spacing_m,component,sample_rate_hz,geometry_status,geometry_note,run_id
0,GEOM_GEODE_T1_1M_REFRACTION_F3001,GEODE_T1_1M_REFRACTION_F3001,geode,T1,T1,T1_1m_refraction,NaN,NaN,None,NaN,Z,NaN,streamer_geometry_pending,Streamer receiver positions require roll/sprea...,run_20260612T200044Z_bdd2ebb7
1,GEOM_GEODE_T1_1M_REFRACTION_F3002,GEODE_T1_1M_REFRACTION_F3002,geode,T1,T1,T1_1m_refraction,NaN,NaN,None,NaN,Z,NaN,streamer_geometry_pending,Streamer receiver positions require roll/sprea...,run_20260612T200044Z_bdd2ebb7
2,GEOM_GEODE_T1_1M_REFRACTION_F3003,GEODE_T1_1M_REFRACTION_F3003,geode,T1,T1,T1_1m_refraction,NaN,NaN,None,NaN,Z,NaN,streamer_geometry_pending,Streamer receiver positions require roll/sprea...,run_20260612T200044Z_bdd2ebb7
3,GEOM_GEODE_T1_1M_REFRACTION_F3004,GEODE_T1_1M_REFRACTION_F3004,geode,T1,T1,T1_1m_refraction,NaN,NaN,None,NaN,Z,NaN,streamer_geometry_pending,Streamer receiver positions require roll/sprea...,run_20260612T200044Z_bdd2ebb7
4,GEOM_GEODE_T1_1M_REFRACTION_F3005,GEODE_T1_1M_REFRACTION_F3005,geode,T1,T1,T1_1m_refraction,1.0,87.0,None,1.0,Z,8000.0,expanded_from_refraction_metadata,"receiver_first_m=87.0, receiver_spacing_m=1.0,...",run_20260612T200044Z_bdd2ebb7


,gather_file_id,event_id,gather_id,instrument_system,line,transect,survey,component,file_type,file_path,source_file_no,n_traces,n_receivers,sample_rate_hz,duration_s,processing_level,geometry_id,run_id
0,GF_GEODE_T1_1M_REFRACTION_F3005_RAW_DAT,GEODE_T1_1M_REFRACTION_F3005,GATHER_GEODE_T1_1M_REFRACTION_F3005_RAW,geode,T1,T1,T1_1m_refraction,Z,SEG2,/Volumes/tachyon/LBSSP_DATA/GEODE_DATA/LBSSP_0...,3005,72,72,8000.0,0.399875,RAW,GEOM_GEODE_T1_1M_REFRACTION_F3005,run_20260612T200044Z_bdd2ebb7
1,GF_GEODE_T1_1M_REFRACTION_F3006_RAW_DAT,GEODE_T1_1M_REFRACTION_F3006,GATHER_GEODE_T1_1M_REFRACTION_F3006_RAW,geode,T1,T1,T1_1m_refraction,Z,SEG2,/Volumes/tachyon/LBSSP_DATA/GEODE_DATA/LBSSP_0...,3006,72,72,8000.0,0.399875,RAW,GEOM_GEODE_T1_1M_REFRACTION_F3006,run_20260612T200044Z_bdd2ebb7
2,GF_GEODE_T1_1M_REFRACTION_F3007_RAW_DAT,GEODE_T1_1M_REFRACTION_F3007,GATHER_GEODE_T1_1M_REFRACTION_F3007_RAW,geode,T1,T1,T1_1m_refraction,Z,SEG2,/Volumes/tachyon/LBSSP_DATA/GEODE_DATA/LBSSP_0...,3007,72,72,8000.0,0.399875,RAW,GEOM_GEODE_T1_1M_REFRACTION_F3007,run_20260612T200044Z_bdd2ebb7
3,GF_GEODE_T1_1M_REFRACTION_F3008_RAW_DAT,GEODE_T1_1M_REFRACTION_F3008,GATHER_GEODE_T1_1M_REFRACTION_F3008_RAW,geode,T1,T1,T1_1m_refraction,Z,SEG2,/Volumes/tachyon/LBSSP_DATA/GEODE_DATA/LBSSP_0...,3008,72,72,8000.0,0.399875,RAW,GEOM_GEODE_T1_1M_REFRACTION_F3008,run_20260612T200044Z_bdd2ebb7
4,GF_GEODE_T1_1M_REFRACTION_F3009_RAW_DAT,GEODE_T1_1M_REFRACTION_F3009,GATHER_GEODE_T1_1M_REFRACTION_F3009_RAW,geode,T1,T1,T1_1m_refraction,Z,SEG2,/Volumes/tachyon/LBSSP_DATA/GEODE_DATA/LBSSP_0...,3009,72,72,8000.0,0.399875,RAW,GEOM_GEODE_T1_1M_REFRACTION_F3009,run_20260612T200044Z_bdd2ebb7


,trace_index_id,event_id,gather_id,instrument_system,line,transect,survey,source_x_m,receiver_index,receiver_x_m,offset_m,component,file_path,trace_number_in_file,sample_rate_hz,npts,amplitude_scale,run_id
0,TR_GEODE_T1_1M_REFRACTION_F3005_001,GEODE_T1_1M_REFRACTION_F3005,GATHER_GEODE_T1_1M_REFRACTION_F3005_RAW,geode,T1,T1,T1_1m_refraction,82.5,1,87.0,4.5,Z,/Volumes/tachyon/LBSSP_DATA/GEODE_DATA/LBSSP_0...,1,8000.0,3199,None,run_20260612T200044Z_bdd2ebb7
1,TR_GEODE_T1_1M_REFRACTION_F3005_002,GEODE_T1_1M_REFRACTION_F3005,GATHER_GEODE_T1_1M_REFRACTION_F3005_RAW,geode,T1,T1,T1_1m_refraction,82.5,2,88.0,5.5,Z,/Volumes/tachyon/LBSSP_DATA/GEODE_DATA/LBSSP_0...,2,8000.0,3199,None,run_20260612T200044Z_bdd2ebb7
2,TR_GEODE_T1_1M_REFRACTION_F3005_003,GEODE_T1_1M_REFRACTION_F3005,GATHER_GEODE_T1_1M_REFRACTION_F3005_RAW,geode,T1,T1,T1_1m_refraction,82.5,3,89.0,6.5,Z,/Volumes/tachyon/LBSSP_DATA/GEODE_DATA/LBSSP_0...,3,8000.0,3199,None,run_20260612T200044Z_bdd2ebb7
3,TR_GEODE_T1_1M_REFRACTION_F3005_004,GEODE_T1_1M_REFRACTION_F3005,GATHER_GEODE_T1_1M_REFRACTION_F3005_RAW,geode,T1,T1,T1_1m_refraction,82.5,4,90.0,7.5,Z,/Volumes/tachyon/LBSSP_DATA/GEODE_DATA/LBSSP_0...,4,8000.0,3199,None,run_20260612T200044Z_bdd2ebb7
4,TR_GEODE_T1_1M_REFRACTION_F3005_005,GEODE_T1_1M_REFRACTION_F3005,GATHER_GEODE_T1_1M_REFRACTION_F3005_RAW,geode,T1,T1,T1_1m_refraction,82.5,5,91.0,8.5,Z,/Volumes/tachyon/LBSSP_DATA/GEODE_DATA/LBSSP_0...,5,8000.0,3199,None,run_20260612T200044Z_bdd2ebb7


## 9. Write catalog rows to SQLite

In [11]:
write_df(conn, shot_events_df, "shot_events", if_exists="append")
write_df(conn, receiver_geometry_df, "receiver_geometry", if_exists="append")
write_df(conn, shot_gather_files_df, "shot_gather_files", if_exists="append")
write_df(conn, trace_index_df, "trace_index", if_exists="append")

conn.commit()

shot_events: wrote 327 rows
receiver_geometry: wrote 7285 rows
shot_gather_files: wrote 185 rows
trace_index: wrote 7056 rows


## 10. Validation queries

In [12]:
q = """
SELECT metadata_source_sheet, survey_type, COUNT(*) AS n_events,
       COUNT(shot_time_utc) AS n_with_time,
       COUNT(geode_file_path) AS n_with_file,
       MIN(source_x_m) AS min_source_x_m,
       MAX(source_x_m) AS max_source_x_m
FROM shot_events
WHERE run_id = ?
GROUP BY metadata_source_sheet, survey_type
ORDER BY metadata_source_sheet
"""
display(pd.read_sql(q, conn, params=(RUN_ID,)))

q = """
SELECT line, ROUND(source_x_m, 2) AS source_x_m, COUNT(*) AS n_events,
       GROUP_CONCAT(DISTINCT survey) AS surveys,
       GROUP_CONCAT(DISTINCT source_type) AS source_types
FROM shot_events
WHERE run_id = ? AND source_x_m IS NOT NULL
GROUP BY line, ROUND(source_x_m, 2)
HAVING COUNT(*) > 1
ORDER BY line, source_x_m
"""
common_positions = pd.read_sql(q, conn, params=(RUN_ID,))
display(common_positions.head(50))
print("Common/repeated shot-position rows:", len(common_positions))

,metadata_source_sheet,survey_type,n_events,n_with_time,n_with_file,min_source_x_m,max_source_x_m
0,T1A_Streamer_MASW,streamer_masw,87,87,87,106.0,230.0
1,T1_1m_Refraction,refraction,46,42,42,82.5,162.5
2,T1_2m_Refraction,refraction,42,42,42,43.0,203.0
3,T1_Streamer_MASW,streamer_masw,83,0,0,87.0,210.0
4,T3_1m_Refraction,refraction,39,0,0,-0.5,75.5
5,T4_1m_Refraction,refraction,30,14,14,-2.5,71.5


,line,source_x_m,n_events,surveys,source_types
0,T1,47.0,2,T1_2m_refraction,hammer
1,T1,82.5,5,T1_1m_refraction,hammer
2,T1,87.0,2,"T1_2m_refraction,Streamer/MASW main transect","hammer,PEG"
3,T1,88.5,2,"T1_1m_refraction,Streamer/MASW main transect","hammer,PEG"
4,T1,94.5,2,"T1_1m_refraction,Streamer/MASW main transect","hammer,PEG"
5,T1,99.0,2,"T1_2m_refraction,Streamer/MASW main transect","hammer,PEG"
6,T1,100.5,2,"T1_1m_refraction,Streamer/MASW main transect","hammer,PEG"
7,T1,102.5,2,T1_1m_refraction,hammer
8,T1,106.5,2,"T1_1m_refraction,Streamer/MASW main transect","hammer,PEG"
9,T1,111.0,2,"T1_2m_refraction,Streamer/MASW main transect","hammer,PEG"


Common/repeated shot-position rows: 27


## 11. Export CSV snapshots for inspection

In [13]:
for table in [
    "processing_runs",
    "transect_crosswalk",
    "acquisition_summary",
    "survey_issues",
    "geode_file_times",
    "shot_events",
    "receiver_geometry",
    "shot_gather_files",
    "trace_index",
]:
    export_table(conn, table, EXPORT_DIR)

print("CSV snapshots written to:", EXPORT_DIR)

exported processing_runs: /Volumes/tachyon/LBSSP_DATA/catalog/catalog_exports/processing_runs.csv
exported transect_crosswalk: /Volumes/tachyon/LBSSP_DATA/catalog/catalog_exports/transect_crosswalk.csv
exported acquisition_summary: /Volumes/tachyon/LBSSP_DATA/catalog/catalog_exports/acquisition_summary.csv
exported survey_issues: /Volumes/tachyon/LBSSP_DATA/catalog/catalog_exports/survey_issues.csv
exported geode_file_times: /Volumes/tachyon/LBSSP_DATA/catalog/catalog_exports/geode_file_times.csv
exported shot_events: /Volumes/tachyon/LBSSP_DATA/catalog/catalog_exports/shot_events.csv
exported receiver_geometry: /Volumes/tachyon/LBSSP_DATA/catalog/catalog_exports/receiver_geometry.csv
exported shot_gather_files: /Volumes/tachyon/LBSSP_DATA/catalog/catalog_exports/shot_gather_files.csv
exported trace_index: /Volumes/tachyon/LBSSP_DATA/catalog/catalog_exports/trace_index.csv
CSV snapshots written to: /Volumes/tachyon/LBSSP_DATA/catalog/catalog_exports


## 12. Useful follow-up queries

In [14]:
q = """
SELECT event_id, line, survey, survey_type, file_no, shot_no, source_x_m,
       source_type, n_blows, n_shots, operator, plate_type,
       shot_time_local, shot_time_utc, geode_file_path
FROM shot_events
WHERE run_id = ?
  AND line = 'T1'
  AND source_x_m IS NOT NULL
ORDER BY source_x_m, shot_time_utc
"""
t1_events = pd.read_sql(q, conn, params=(RUN_ID,))
display(t1_events.head(100))

q = """
SELECT event_id, line, survey, file_no, source_x_m, source_type, comment
FROM shot_events
WHERE run_id = ?
  AND (
      LOWER(COALESCE(source_type,'')) LIKE '%betsy%'
      OR LOWER(COALESCE(comment,'')) LIKE '%betsy%'
      OR LOWER(COALESCE(source_type,'')) LIKE '%peg%'
      OR LOWER(COALESCE(comment,'')) LIKE '%peg%'
  )
ORDER BY line, file_no
"""
display(pd.read_sql(q, conn, params=(RUN_ID,)))

,event_id,line,survey,survey_type,file_no,shot_no,source_x_m,source_type,n_blows,n_shots,operator,plate_type,shot_time_local,shot_time_utc,geode_file_path
0,GEODE_T1_2M_REFRACTION_F3047,T1,T1_2m_refraction,refraction,3047,1,43.0,hammer,10.0,NaN,Saheed,metal,2026-05-18T20:18:44,2026-05-19T00:18:44+00:00,/Volumes/tachyon/LBSSP_DATA/GEODE_DATA/LBSSP_0...
1,GEODE_T1_2M_REFRACTION_F3048,T1,T1_2m_refraction,refraction,3048,2,47.0,hammer,10.0,NaN,Izzy,metal,2026-05-18T20:21:44,2026-05-19T00:21:44+00:00,/Volumes/tachyon/LBSSP_DATA/GEODE_DATA/LBSSP_0...
2,GEODE_T1_2M_REFRACTION_F3088,T1,T1_2m_refraction,refraction,3088,42,47.0,hammer,1.0,NaN,Betsy,metal,2026-05-18T23:12:53,2026-05-19T03:12:53+00:00,/Volumes/tachyon/LBSSP_DATA/GEODE_DATA/LBSSP_0...
3,GEODE_T1_2M_REFRACTION_F3049,T1,T1_2m_refraction,refraction,3049,3,51.0,hammer,10.0,NaN,Yuly,metal,2026-05-18T20:26:16,2026-05-19T00:26:16+00:00,/Volumes/tachyon/LBSSP_DATA/GEODE_DATA/LBSSP_0...
4,GEODE_T1_2M_REFRACTION_F3050,T1,T1_2m_refraction,refraction,3050,4,55.0,hammer,10.0,NaN,Pati,metal,2026-05-18T20:30:19,2026-05-19T00:30:19+00:00,/Volumes/tachyon/LBSSP_DATA/GEODE_DATA/LBSSP_0...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,GEODE_T1_STREAMER_MASW_F1036,T1,Streamer/MASW main transect,streamer_masw,1036,36,139.5,PEG,NaN,4.0,None,None,None,None,None
96,GEODE_T1_1M_REFRACTION_F3035,T1,T1_1m_refraction,refraction,3035,35,140.5,hammer,10.0,NaN,Brian,metal,2026-05-18T18:05:21,2026-05-18T22:05:21+00:00,/Volumes/tachyon/LBSSP_DATA/GEODE_DATA/LBSSP_0...
97,GEODE_T1_STREAMER_MASW_F1037,T1,Streamer/MASW main transect,streamer_masw,1037,37,141.0,PEG,NaN,4.0,None,None,None,None,None
98,GEODE_T1_STREAMER_MASW_F1038,T1,Streamer/MASW main transect,streamer_masw,1038,38,142.5,PEG,NaN,4.0,None,None,None,None,None


,event_id,line,survey,file_no,source_x_m,source_type,comment
0,GEODE_T1_STREAMER_MASW_F1001,T1,Streamer/MASW main transect,1001,87.0,PEG,145 [ft] is location of shot relative to last ...
1,GEODE_T1_STREAMER_MASW_F1002,T1,Streamer/MASW main transect,1002,88.5,PEG,None
2,GEODE_T1_STREAMER_MASW_F1003,T1,Streamer/MASW main transect,1003,90.0,PEG,None
3,GEODE_T1_STREAMER_MASW_F1004,T1,Streamer/MASW main transect,1004,91.5,PEG,None
4,GEODE_T1_STREAMER_MASW_F1005,T1,Streamer/MASW main transect,1005,93.0,PEG,None
...,...,...,...,...,...,...,...
166,GEODE_T1A_STREAMER_MASW_F2083,T1A,Streamer/MASW western transect,2083,227.0,PEG,single file_no match
167,GEODE_T1A_STREAMER_MASW_F2084,T1A,Streamer/MASW western transect,2084,228.5,PEG,single file_no match
168,GEODE_T1A_STREAMER_MASW_F2085,T1A,Streamer/MASW western transect,2085,230.0,PEG,single file_no match
169,GEODE_T1A_STREAMER_MASW_F2086,T1A,Streamer/MASW western transect,2086,230.0,hammer jb,PEG-hammer comparison; same number of hammer b...
